# Model-agnostic Gateway

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/pl/11_1_Model_agnostic_gateway_production.ipynb)

| | |
|---|---|
| 🎯 **Czego się nauczysz** | Wzorce produkcyjne: akceptacja human-in-the-loop przez middleware i niezależna od modelu bramka API w FastAPI. |
| 🔑 **Klucze API** | `OPENAI_API_KEY` |
| 💲 **Koszt API** | bardzo niski – kilka krótkich zapytań |
| ⏱️ **Czas wykonania** | < 10 s |

### Instalacja bibliotek

In [1]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
# Pomijane, gdy brak pip, np. w środowisku `uv sync`, które ma je już zainstalowane (zob. README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU langchain langgraph langchain-openai fastapi uvicorn python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### Human in the loop

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

@tool
def risky_operation(secret: str) -> str:
    """Perform a risky operation that requires manual approval."""
    return f"Executed risky operation with: {secret}"

tools = [risky_operation]
model = ChatOpenAI(model="gpt-6-luna")

hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "risky_operation": {"allowed_decisions": ["approve", "edit", "reject"]}
    },
    description_prefix="Manual approval required for risky operation:"
)


checkpointer = MemorySaver()
agent = create_agent(
    model=model,
    tools=tools,
    middleware=[hitl],
    checkpointer=checkpointer,
    debug=True
)

config = {"configurable": {"thread_id": "hitl-demo-1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Please run the risky operation with secret code $%45654@."}]},
    config=config,
)



[values] {'messages': [HumanMessage(content='Please run the risky operation with secret code $%45654@.', additional_kwargs={}, response_metadata={}, id='a10db371-ba77-4220-a4b8-9bd647b32323')]}
[updates] {'model': {'messages': [AIMessage(content=[{'arguments': '{"secret":"$%45654@"}', 'call_id': 'call_c8xAYbvdp9X7Bjc9f72wvNHZ', 'name': 'risky_operation', 'type': 'function_call', 'id': 'fc_00aafea16141b859006ad7b259b7a887d2b3d2f1ae2e2aa8d6', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_00aafea16141b859006ad7b25890c887d2988d39011c6bdd87', 'created_at': 1792520792.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_00aafea16141b859006ad7b25890c887d2988d39011c6bdd87', tool_calls=[{'name': 'risky_operation', 'args': {'secret': '$%45654@'}, 'id': 'call_c8xAYbvdp9X7Bjc9f72wvNHZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met

In [4]:
if "__interrupt__" in result:
    print("Interrupt detected!")
    decisions = [{"type": "approve"}]

    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
    )


Interrupt detected!
[values] {'messages': [HumanMessage(content='Please run the risky operation with secret code $%45654@.', additional_kwargs={}, response_metadata={}, id='a10db371-ba77-4220-a4b8-9bd647b32323'), AIMessage(content=[{'arguments': '{"secret":"$%45654@"}', 'call_id': 'call_c8xAYbvdp9X7Bjc9f72wvNHZ', 'name': 'risky_operation', 'type': 'function_call', 'id': 'fc_00aafea16141b859006ad7b259b7a887d2b3d2f1ae2e2aa8d6', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_00aafea16141b859006ad7b25890c887d2988d39011c6bdd87', 'created_at': 1792520792.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_00aafea16141b859006ad7b25890c887d2988d39011c6bdd87', tool_calls=[{'name': 'risky_operation', 'args': {'secret': '$%45654@'}, 'id': 'call_c8xAYbvdp9X7Bjc9f72wvNHZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t

### Model agnostic API gateway

In [5]:
# Umieść poniższy kod w pliku app.py

from fastapi import FastAPI, Header
from pydantic import BaseModel
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_openai import ChatOpenAI

class ChatRequest(BaseModel):
    message: str

class ChatResponse(BaseModel):
    provider: str
    model: str
    answer: str

prompt = ChatPromptTemplate.from_messages([
    ("system", "Jesteś pomocnym asystentem."),
    ("human", "{message}")
])

def build_model(x_model: str):
    """
    x_model format:
      - 'openai:gpt-6-luna'
    """
    if ":" in x_model:
        provider, model_name = x_model.split(":", 1)
    else:
        provider, model_name = "openai", x_model

    provider = provider.lower().strip()

    if provider == "openai":
        return provider, model_name, ChatOpenAI(model=model_name, temperature=0)

    # if provider == "anthropic": # wsparcie dla kolejnego providera LLM API
    #     return provider, model_name, ChatAnthropic(model=model_name, temperature=0)

    def _unknown(inputs: dict):
        return AIMessage(content=f"(unknown provider) Echo: {inputs.get('message','')}")
    return "unknown", x_model, RunnableLambda(_unknown)


app = FastAPI(title="Model-Agnostic LangChain Gateway")


@app.post("/chat", response_model=ChatResponse)
def chat_endpoint(
    req: ChatRequest,
    x_model: str = Header(default="openai:gpt-6-luna", alias="X-Model"),
):
    provider, model_name, model = build_model(x_model)
    chain = prompt | model | StrOutputParser()
    answer: str = chain.invoke({"message": req.message})
    return ChatResponse(provider=provider, model=model_name, answer=answer)

1. Umieść powyższy kod w pliku `app.py`
2. Uruchom serwer:
`uvicorn app:app --reload`

3. Wyślij request
```
curl -X POST 'http://127.0.0.1:8000/chat' \
  -H 'Content-Type: application/json' \
  -H 'X-Model: openai:gpt-5-mini' \
  -d '{"message":"Podaj 3 zalety Pythona."}'
```

```
curl -X POST 'http://127.0.0.1:8000/chat' \
  -H 'Content-Type: application/json' \
  -H 'X-Model: openai:gpt-6-luna' \
  -d '{"message":"Podaj 3 zalety Pythona."}'
```